# Global Cybersecurity Threats Analysis (2015–2024)

**Goal:** Analyze 3,000 cybersecurity incidents to understand trends, the most damaging attack types, the most targeted industries and countries, and which defenses are linked to faster resolution.

**Dataset:** [Global Cybersecurity Threats 2015–2024](https://www.kaggle.com/datasets/atharvasoundankar/global-cybersecurity-threats-2015-2024) (Kaggle, CC0 Public Domain)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/Global_Cybersecurity_Threats_2015-2024.csv")
df.head()

,Country,Year,Attack Type,Target Industry,Financial Loss (in Million $),Number of Affected Users,Attack Source,Security Vulnerability Type,Defense Mechanism Used,Incident Resolution Time (in Hours)
0,China,2019,Phishing,Education,80.53,773169,Hacker Group,Unpatched Software,VPN,63
1,China,2019,Ransomware,Retail,62.19,295961,Hacker Group,Unpatched Software,Firewall,71
2,India,2017,Man-in-the-Middle,IT,38.65,605895,Hacker Group,Weak Passwords,VPN,20
3,UK,2024,Ransomware,Telecommunications,41.44,659320,Nation-state,Social Engineering,AI-based Detection,7
4,Germany,2018,Man-in-the-Middle,IT,74.41,810682,Insider,Social Engineering,VPN,68


## 1. Data Exploration

In [2]:
df.shape

(3000, 10)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Country                              3000 non-null   str    
 1   Year                                 3000 non-null   int64  
 2   Attack Type                          3000 non-null   str    
 3   Target Industry                      3000 non-null   str    
 4   Financial Loss (in Million $)        3000 non-null   float64
 5   Number of Affected Users             3000 non-null   int64  
 6   Attack Source                        3000 non-null   str    
 7   Security Vulnerability Type          3000 non-null   str    
 8   Defense Mechanism Used               3000 non-null   str    
 9   Incident Resolution Time (in Hours)  3000 non-null   int64  
dtypes: float64(1), int64(3), str(6)
memory usage: 234.5 KB


In [4]:
df.describe()

,Year,Financial Loss (in Million $),Number of Affected Users,Incident Resolution Time (in Hours)
count,3000.000000,3000.000000,3000.000000,3000.000000
mean,2019.570333,50.492970,504684.136333,36.476000
std,2.857932,28.791415,289944.084972,20.570768
min,2015.000000,0.500000,424.000000,1.000000
25%,2017.000000,25.757500,255805.250000,19.000000
50%,2020.000000,50.795000,504513.000000,37.000000
75%,2022.000000,75.630000,758088.500000,55.000000
max,2024.000000,99.990000,999635.000000,72.000000


In [5]:
df.nunique()

Country                                  10
Year                                     10
Attack Type                               6
Target Industry                           7
Financial Loss (in Million $)          2536
Number of Affected Users               2998
Attack Source                             4
Security Vulnerability Type               4
Defense Mechanism Used                    5
Incident Resolution Time (in Hours)      72
dtype: int64

In [6]:
df.isnull().sum()

Country                                0
Year                                   0
Attack Type                            0
Target Industry                        0
Financial Loss (in Million $)          0
Number of Affected Users               0
Attack Source                          0
Security Vulnerability Type            0
Defense Mechanism Used                 0
Incident Resolution Time (in Hours)    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

### Exploration Summary
- The dataset has **3,000 rows** (incidents) and **10 columns**.
- **Missing values:** none, all columns are complete.
- **Duplicate rows:** none.
- **6 text columns:** Country, Attack Type, Target Industry, Attack Source, Security Vulnerability Type, Defense Mechanism Used.
- **4 numeric columns:** Year, Financial Loss (in Million $), Number of Affected Users, Incident Resolution Time (in Hours).
- **Categories are small:** 10 countries, 6 attack types, 7 industries, 4 attack sources, 4 vulnerability types, 5 defense mechanisms. This makes them suitable for grouping and comparison.
- **Note:** Financial loss (0.5–100M $), affected users and resolution time (1–72 hours) are spread almost perfectly evenly between their min and max. Real incident data is usually skewed (many small incidents, few huge ones), so this dataset is likely **synthetic**. Differences between groups are expected to be small, and findings should be interpreted with care.

## 2. Data Cleaning

**Rename columns:** The original names are long and contain spaces and symbols (e.g. `Financial Loss (in Million $)`). Short snake_case names are easier to type and less error-prone in code.

In [8]:
df = df.rename(columns={
    "Country": "country",
    "Year": "year",
    "Attack Type": "attack_type",
    "Target Industry": "target_industry",
    "Financial Loss (in Million $)": "financial_loss_musd",
    "Number of Affected Users": "affected_users",
    "Attack Source": "attack_source",
    "Security Vulnerability Type": "vulnerability_type",
    "Defense Mechanism Used": "defense_mechanism",
    "Incident Resolution Time (in Hours)": "resolution_hours",
})
df.columns

Index(['country', 'year', 'attack_type', 'target_industry',
       'financial_loss_musd', 'affected_users', 'attack_source',
       'vulnerability_type', 'defense_mechanism', 'resolution_hours'],
      dtype='str')

**Check text columns:** Look at every unique value to catch typos, inconsistent capitalization, or extra spaces (e.g. "UK" vs "uk " would be counted as two countries).

In [9]:
text_cols = ["country", "attack_type", "target_industry",
             "attack_source", "vulnerability_type", "defense_mechanism"]

for col in text_cols:
    print(col, ":", sorted(df[col].unique()))

country : ['Australia', 'Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'Russia', 'UK', 'USA']
attack_type : ['DDoS', 'Malware', 'Man-in-the-Middle', 'Phishing', 'Ransomware', 'SQL Injection']
target_industry : ['Banking', 'Education', 'Government', 'Healthcare', 'IT', 'Retail', 'Telecommunications']
attack_source : ['Hacker Group', 'Insider', 'Nation-state', 'Unknown']
vulnerability_type : ['Social Engineering', 'Unpatched Software', 'Weak Passwords', 'Zero-day']
defense_mechanism : ['AI-based Detection', 'Antivirus', 'Encryption', 'Firewall', 'VPN']


**Strip whitespace:** No problems were visible, but invisible leading/trailing spaces are a common issue. Stripping them is a safe, defensive step.

In [10]:
for col in text_cols:
    df[col] = df[col].str.strip()

**Convert text columns to `category`:** Each text column has only 4–10 repeated values. The `category` type stores them more efficiently and tells pandas (and the reader) that these are groups, not free text.

In [11]:
for col in text_cols:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   country              3000 non-null   category
 1   year                 3000 non-null   int64   
 2   attack_type          3000 non-null   category
 3   target_industry      3000 non-null   category
 4   financial_loss_musd  3000 non-null   float64 
 5   affected_users       3000 non-null   int64   
 6   attack_source        3000 non-null   category
 7   vulnerability_type   3000 non-null   category
 8   defense_mechanism    3000 non-null   category
 9   resolution_hours     3000 non-null   int64   
dtypes: category(6), float64(1), int64(3)
memory usage: 111.7 KB


**Check value ranges:** Financial loss, affected users, and resolution time should never be negative or zero, and years should be within 2015–2024.

In [12]:
print("Negative losses:", (df["financial_loss_musd"] < 0).sum())
print("Negative affected users:", (df["affected_users"] < 0).sum())
print("Resolution time <= 0:", (df["resolution_hours"] <= 0).sum())
print("Years:", df["year"].min(), "to", df["year"].max())

Negative losses: 0
Negative affected users: 0
Resolution time <= 0: 0
Years: 2015 to 2024


**Save cleaned dataset:** Saved as a separate file so the raw data stays untouched, and so Project 2 (Interactive Dashboard) can reuse it directly.

In [13]:
df.to_csv("../data/cybersecurity_threats_clean.csv", index=False)

### Cleaning Summary
- Renamed all columns to short snake_case names.
- Checked all text values: no typos or inconsistent labels found; stripped whitespace as a precaution.
- Converted 6 text columns to `category` type (reduced memory usage).
- No missing values or duplicates → no rows removed.
- No negative or impossible values; years range from 2015 to 2024.
- Saved the clean data to `data/cybersecurity_threats_clean.csv`.